# Cascade analysis

Analysis of the cascaded 3-head MLP (`mlp.py`): defect -> joint mechanism -> risk.
Every plot is driven by the already-trained checkpoint -- no training happens here.

Produce the checkpoint once from the CLI, then run this notebook top to bottom:

```
python mlp.py --model-out results/mlp_model.pt
```


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix

# generator.py and mlp.py are self-contained in the repo root
sys.path.insert(0, str(Path.cwd()))

from mlp import (defect_names, encode, evaluate, load_model,
                 load_spec, param_ids)

DEFECT_HEX = {"no_defect": "#8a8d91", "open_circuit": "#2f6db5",
              "solder_bridging": "#c1432e"}
SHORT_DEF = {"no_defect": "none", "open_circuit": "open", "solder_bridging": "bridge"}
SHORT_MECH = {"no_mechanism": "none", "aperture_overfill": "ap.overfill",
              "poor_paste_transfer": "poor.transfer",
              "reflow_spreading": "rfl.spread", "non_coalescence": "non.coal"}


def short_joint(label):
    """'<printing>__<reflow>' -> 'printing/reflow' with short mechanism names."""
    p, r = label.split("__")
    return f"{SHORT_MECH.get(p, p)}/{SHORT_MECH.get(r, r)}"


def _pack(model, enc, df, spec, split="test"):
    """Predict on one split and bundle what the plots need (cascade heads)."""
    ids, names = param_ids(spec), defect_names(spec)
    te = np.where(df["split"].to_numpy() == split)[0]
    Xraw = df.iloc[te][ids].to_numpy(np.float32)
    X = ((Xraw - enc["mu"]) / enc["sd"]).astype(np.float32)
    pred = model.predict(torch.from_numpy(X))
    return {
        "te": te, "ids": ids, "names": names, "vocab": enc["mech_vocab"],
        "y_def": enc["y_def"][te], "d_pred": pred["defect_argmax"].cpu().numpy(),
        "defect_prob": pred["defect_prob"].cpu().numpy(),
        "post": df.iloc[te][[f"p_{n}" for n in names]].to_numpy(),
        "y_mech": enc["y_mech"][te], "m_pred": pred["mech_argmax"].cpu().numpy(),
        "mech_prob": pred["mech_prob"].cpu().numpy(),
        "risk_pred": pred["risk_prob"].cpu().numpy(), "y_risk": enc["y_risk"][te],
    }

In [ ]:
# No training here -- produce the checkpoint once from the CLI:
#   python mlp.py --model-out results/mlp_model.pt
spec = load_spec("domain/smt_paper.yaml")
df = pd.read_csv("data/smt_synthetic.csv")
figs = Path("figs"); figs.mkdir(exist_ok=True)

model, ckpt = load_model("results/mlp_model.pt")
enc = encode(df, spec)
enc["mu"], enc["sd"] = ckpt["standardize"]["mu"], ckpt["standardize"]["sd"]
enc["val_loss_history"] = ckpt.get("val_loss_history", [])
enc["best_epoch"] = ckpt.get("best_epoch", 0)
metrics = evaluate(model, enc, df, spec, model_version=ckpt["model_version"])
pack = _pack(model, enc, df, spec)
bayes = metrics["bayes_optimal_accuracy"]
print("loaded", metrics["model_version"])
print(f"defect acc {metrics['defect_head']['accuracy']:.4f}  bayes {bayes:.4f}  "
      f"mech joint {metrics['mechanism_accuracy']['joint_accuracy']:.4f}  "
      f"risk MAE {metrics['risk_mae']:.4f}")

In [ ]:
# head1 defect: row-normalized confusion -- cascade vs the Bayes oracle.
short = [SHORT_DEF[n] for n in pack["names"]]
panels = [("cascade defect head", pack["d_pred"], pack["y_def"]),
          ("Bayes oracle", pack["post"].argmax(1), pack["y_def"])]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4))
for ax, (title, pred, yt) in zip(axes, panels):
    cm = confusion_matrix(yt, pred, labels=[0, 1, 2]).astype(float)
    cmn = cm / cm.sum(1, keepdims=True)
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(short)
    ax.set_yticks(range(3)); ax.set_yticklabels(short)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.5 else "#333", fontsize=9)
fig.suptitle("Defect head confusion (row-normalized recall)")
fig.tight_layout(); fig.savefig(figs / "defect_confusion.png", dpi=140); plt.show()

In [ ]:
# head1 defect metrics vs the Bayes ceiling and the paper's reported accuracy.
d = metrics["defect_head"]
labels = ["accuracy", "weighted F1", "macro F1"]
vals = [d["accuracy"], d["weighted_f1"], d["macro_f1"]]
fig, ax = plt.subplots(figsize=(7, 4.4))
bars = ax.bar(labels, vals, color="#2f6db5", width=0.55)
ax.axhline(bayes, color="#c1432e", ls="--", lw=1.2, label=f"Bayes ceiling {bayes:.3f}")
ax.axhline(metrics["paper_reference"]["accuracy"], color="#6aa84f", ls=":", lw=1.2,
           label=f"paper acc {metrics['paper_reference']['accuracy']:.3f}")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)
ax.set_ylim(0.8, 1.0); ax.set_ylabel("score"); ax.set_title("Defect head (test split)")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(figs / "defect_metrics.png", dpi=140); plt.show()

In [ ]:
# head2 mechanism: joint confusion (only the classes that actually occur shown).
vocab = pack["vocab"]
present = sorted(set(pack["y_mech"].tolist()) | set(pack["m_pred"].tolist()))
labels_short = [short_joint(vocab[i]) for i in present]
cm = confusion_matrix(pack["y_mech"], pack["m_pred"], labels=present).astype(float)
cmn = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(present))); ax.set_xticklabels(labels_short, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(present))); ax.set_yticklabels(labels_short, fontsize=8)
ax.set_xlabel("predicted joint mechanism"); ax.set_ylabel("true joint mechanism")
ax.set_title(f"Mechanism head: joint {len(present)}-class confusion (of 9 possible)")
for i in range(len(present)):
    for j in range(len(present)):
        if cmn[i, j] > 0.01:
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.5 else "#333", fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout(); fig.savefig(figs / "mechanism_confusion.png", dpi=140); plt.show()

In [ ]:
# head2 decomposed: split each predicted/true joint label back into its two stages.
vocab = pack["vocab"]
pred_pairs = np.array([vocab[i].split("__") for i in pack["m_pred"]])
true_pairs = np.array([vocab[i].split("__") for i in pack["y_mech"]])
stages = list(spec["mechanisms"].keys())
acc = {
    "joint (9-class)": metrics["mechanism_accuracy"]["joint_accuracy"],
    stages[0]: float((pred_pairs[:, 0] == true_pairs[:, 0]).mean()),
    stages[1]: float((pred_pairs[:, 1] == true_pairs[:, 1]).mean()),
}
fig, ax = plt.subplots(figsize=(7, 4.4))
bars = ax.bar(list(acc), list(acc.values()), color=["#444", "#2f6db5", "#6aa84f"], width=0.55)
for b, v in zip(bars, acc.values()):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.004, f"{v:.3f}", ha="center", fontsize=9)
ax.set_ylim(0.85, 1.0); ax.set_ylabel("accuracy")
ax.set_title("Mechanism: joint head vs its per-stage decomposition")
fig.tight_layout(); fig.savefig(figs / "mechanism_perstage.png", dpi=140); plt.show()

In [ ]:
# head3 risk: reliability -- bin by predicted risk, plot the mean TRUE graded risk.
pr = pack["risk_pred"].ravel(); tr = pack["y_risk"].ravel()
edges = np.linspace(0, 1, 21); idx = np.clip(np.digitize(pr, edges) - 1, 0, len(edges) - 2)
centers, mean_true = [], []
for b in range(len(edges) - 1):
    m = idx == b
    if m.any():
        centers.append((edges[b] + edges[b + 1]) / 2); mean_true.append(tr[m].mean())
fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot([0, 1], [0, 1], color="#999", ls="--", lw=1, label="perfect calibration")
ax.plot(centers, mean_true, "o-", color="#2f6db5", lw=2, label="cascade risk head")
ax.set_xlabel("predicted risk (binned)"); ax.set_ylabel("mean true graded risk")
ax.set_title(f"Risk head reliability  (overall MAE {metrics['risk_mae']:.4f})")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(figs / "risk_reliability.png", dpi=140); plt.show()

In [ ]:
# head3 risk: per-parameter mean absolute error.
ids = pack["ids"]
mae = np.abs(pack["risk_pred"] - pack["y_risk"]).mean(0)
order = np.argsort(mae)
fig, ax = plt.subplots(figsize=(8, 4.6))
bars = ax.barh([ids[i] for i in order], mae[order], color="#2f6db5")
for b, v in zip(bars, mae[order]):
    ax.text(v + 0.0002, b.get_y() + b.get_height() / 2, f"{v:.4f}", va="center", fontsize=8)
ax.set_xlabel("mean absolute error"); ax.set_title("Risk head MAE per parameter")
fig.tight_layout(); fig.savefig(figs / "risk_mae_per_param.png", dpi=140); plt.show()

In [ ]:
# Training: validation-loss curve with the early-stopping best epoch marked.
vh = enc["val_loss_history"]; be = enc["best_epoch"]
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.plot(range(len(vh)), vh, "o-", color="#2f6db5", lw=1.8, ms=4, label="val loss")
if vh:
    ax.axvline(be, color="#c1432e", ls="--", lw=1.2, label=f"best epoch {be} ({vh[be]:.4f})")
ax.set_xlabel("epoch"); ax.set_ylabel("validation loss"); ax.set_title("Cascade training curve")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
fig.tight_layout(); fig.savefig(figs / "training_curve.png", dpi=140); plt.show()

In [ ]:
# The decoupling we wanted: head3 still reports per-parameter risk even when head2
# predicts no_mechanism. Compare the per-board MAX risk for boards the mechanism
# head calls no_mechanism vs boards it assigns a real mechanism.
vocab = pack["vocab"]
nm = vocab.index("no_mechanism__no_mechanism")
max_risk = pack["risk_pred"].max(1)
is_nm = pack["m_pred"] == nm
fig, ax = plt.subplots(figsize=(8, 4.6))
bins = np.linspace(0, 1, 41)
ax.hist(max_risk[is_nm], bins=bins, density=True, color="#8a8d91", alpha=0.7,
        label=f"head2 = no_mechanism  (n={int(is_nm.sum()):,})")
ax.hist(max_risk[~is_nm], bins=bins, density=True, color="#c1432e", alpha=0.6,
        label=f"head2 = a mechanism  (n={int((~is_nm).sum()):,})")
ax.axvline(spec["risk_function"]["p_L"], color="#333", ls=":", lw=1,
           label=f"risk floor p_L = {spec['risk_function']['p_L']}")
ax.set_xlabel("max per-parameter risk on the board (head3)"); ax.set_ylabel("density (log)")
ax.set_title("head3 still flags risk under a no_mechanism call (early-warning drift)")
ax.legend(frameon=False, fontsize=8); ax.set_yscale("log")
fig.tight_layout(); fig.savefig(figs / "risk_under_no_mechanism.png", dpi=140); plt.show()